In [1]:
import cv2
import os
import numpy as np
import math
import torch
from src.det_model.model import BasicModel
from src.eval.seg_detector_representer import SegDetectorRepresenter
from shapely.geometry import Polygon
import pyclipper
RGB_MEAN = np.array([122.67891434, 116.66876762, 104.00698793])
IMAGE_SHORT_SIDE=512
RESULT_DIR = 'output'
BOX_THRESH=0.7
THRESH=0.3
MIN_SIZE = 3
MAX_CANDIDATES =100
model = BasicModel()
representer = SegDetectorRepresenter()

In [14]:
state_dict = torch.load("det_checkpoints/ckpt-155/pytorch_model.bin")
model.load_state_dict(state_dict)
model = model.eval()

In [85]:
def resize_image(img):
    height, width, _ = img.shape
    if height < width:
        new_height = IMAGE_SHORT_SIDE
        new_width = int(math.ceil(new_height / height * width / 32) * 32)
    else:
        new_width = IMAGE_SHORT_SIDE
        new_height = int(math.ceil(new_width / width * height / 32) * 32)
    resized_img = cv2.resize(img, (new_width, new_height))
    return resized_img

def load_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR).astype('float32')
    original_shape = img.shape[:2]
    img = resize_image(img)
    img -= RGB_MEAN
    img /= 255.
    img = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0)
    return img, original_shape

def format_output(batch, output):
        batch_boxes, batch_scores = output
        final_boxes = []
        for index in range(batch['image'].shape[0]):
            original_shape = batch['shape'][index]
            filename = batch['filename'][index]
            result_file_name = 'res_' + filename.split('/')[-1].split('.')[0] + '.txt'
            result_file_path = os.path.join(RESULT_DIR, result_file_name)
            boxes = batch_boxes[index]
            scores = batch_scores[index]
            with open(result_file_path, 'wt') as res:
                for i in range(boxes.shape[0]):
                    score = scores[i]
                    if score < BOX_THRESH:
                        continue
                    box = boxes[i,:,:].reshape(-1).tolist()
                    result = ",".join([str(int(x)) for x in box])
                    res.write(result + ',' + str(score) + "\n")
                    final_boxes.append(box)
        return final_boxes

In [86]:
def restore(image):
    image = image.squeeze(0).permute(1, 2, 0).to('cpu').numpy()
    image = image * 255.
    image += RGB_MEAN
    image = image.astype(np.uint8)
    return image

In [104]:
def demo_visualize(image_path, boxes):
    original_image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    pred_canvas = original_image.copy()

    for box in boxes:
        box = np.array(box).astype(np.int32).reshape(-1, 2)
        print(box)
        cv2.polylines(pred_canvas, [box], True, (0, 255, 0), 2)

    return pred_canvas

In [105]:
def get_mini_boxes(contour):
    bounding_box = cv2.minAreaRect(contour)
    points = sorted(list(cv2.boxPoints(bounding_box)), key=lambda x: x[0])

    index_1, index_2, index_3, index_4 = 0, 1, 2, 3
    if points[1][1] > points[0][1]:
        index_1 = 0
        index_4 = 1
    else:
        index_1 = 1
        index_4 = 0
    if points[3][1] > points[2][1]:
        index_2 = 2
        index_3 = 3
    else:
        index_2 = 3
        index_3 = 2

    box = [points[index_1], points[index_2],
            points[index_3], points[index_4]]
    return box, min(bounding_box[1])


def box_score_fast(bitmap, _box):
    h, w = bitmap.shape[:2]
    box = _box.copy()
    xmin = np.clip(np.floor(box[:, 0].min()).astype(int), 0, w - 1)
    xmax = np.clip(np.ceil(box[:, 0].max()).astype(int), 0, w - 1)
    ymin = np.clip(np.floor(box[:, 1].min()).astype(int), 0, h - 1)
    ymax = np.clip(np.ceil(box[:, 1].max()).astype(int), 0, h - 1)

    mask = np.zeros((ymax - ymin + 1, xmax - xmin + 1), dtype=np.uint8)
    box[:, 0] = box[:, 0] - xmin
    box[:, 1] = box[:, 1] - ymin
    cv2.fillPoly(mask, box.reshape(1, -1, 2).astype(np.int32), 1)
    return cv2.mean(bitmap[ymin:ymax+1, xmin:xmax+1], mask)[0]

In [106]:
def unclip(box, unclip_ratio=1.5):
    poly = Polygon(box)
    distance = poly.area * unclip_ratio / poly.length
    offset = pyclipper.PyclipperOffset()
    offset.AddPath(box, pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
    expanded = np.array(offset.Execute(distance))
    return expanded

In [107]:
def boxes_from_bitmap(pred, _bitmap, dest_width, dest_height):
        '''
        _bitmap: single map with shape (1, H, W),
            whose values are binarized as {0, 1}
        '''
        
        assert _bitmap.shape[0] == 1
        bitmap = _bitmap[0]  # The first channel
        pred = pred[0]
        height, width = bitmap.shape
        contours, _ = cv2.findContours(
            (bitmap*255).astype(np.uint8),
            cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        num_contours = min(len(contours), MAX_CANDIDATES)
        boxes = np.zeros((num_contours, 4, 2), dtype=np.int16)
        scores = np.zeros((num_contours,), dtype=np.float32)

        for index in range(num_contours):
            contour = contours[index]
            points, sside = get_mini_boxes(contour)
            if sside < MIN_SIZE:
                continue
            points = np.array(points)
            score = box_score_fast(pred, points.reshape(-1, 2))
            if BOX_THRESH > score:
                continue
        
            box = unclip(points).reshape(-1, 1, 2)
            box, sside = get_mini_boxes(box)
            if sside < MIN_SIZE + 2:
                continue
            box = np.array(box)
            if not isinstance(dest_width, int):
                dest_width = dest_width.item()
                dest_height = dest_height.item()
            
            box[:, 0] = np.clip(
                np.round(box[:, 0] / width * dest_width), 0, dest_width)
            box[:, 1] = np.clip(
                np.round(box[:, 1] / height * dest_height), 0, dest_height)
            boxes[index, :, :] = box.astype(np.int16)
            scores[index] = score
        return boxes, scores

def post_process(batch, pred):
    segmentation = (pred > THRESH).astype(int)
    boxes_batch = []
    scores_batch = []
    images = batch['image']

    for batch_index in range(images.shape[0]):
        height, width = batch['shape'][batch_index]
        boxes, scores = boxes_from_bitmap(
                pred[batch_index],
                segmentation[batch_index], width, height)
        boxes_batch.append(boxes)
        scores_batch.append(scores)
    return boxes_batch, scores_batch

In [108]:
image_paths = ['det_data/TDTR/train_images/IMG_0168.jpg','det_data/TDTR/train_images/IMG_0180.jpg','det_data/TDTR/train_images/IMG_0362.jpg','det_data/TDTR/train_images/IMG_1989.JPG']

In [112]:
image_paths = [os.path.join("det_data/TextOCR/train_images", filename) for filename in os.listdir("det_data/TextOCR/train_images")]

In [113]:
# image_paths = [os.path.join("det_data/TextOCR/train_images", "2ae4b0cc873e2373.jpg")]

In [114]:
for image_path in image_paths:
    batch = dict()
    batch['filename'] = [image_path]
    img, original_shape = load_image(image_path)
    batch['shape'] = [original_shape]
    with torch.no_grad():
        pred = model(img)
        batch['image'] = img.cpu().numpy()
        pred = pred.cpu().numpy()
        output = post_process(batch, pred) 
        if not os.path.isdir(RESULT_DIR):
            os.mkdir(RESULT_DIR)
        final_boxes = format_output(batch, output)
        print(final_boxes)

        vis_image = demo_visualize(image_path, final_boxes)
        cv2.imwrite(os.path.join(RESULT_DIR, image_path.split('/')[-1].split('.')[0]+'.jpg'), vis_image)

[[203, 650, 360, 639, 364, 690, 207, 701]]
[[203 650]
 [360 639]
 [364 690]
 [207 701]]
[]
[]
[]
[]


In [51]:
len(output[0][0])

100

In [52]:
len(output[1][0])

100

In [58]:
output[0][0][1], output[1][0][1]

(array([[0, 0],
        [0, 0],
        [0, 0],
        [0, 0]], dtype=int16),
 np.float32(0.0))

In [59]:
for i in range(100):
    if output[1][0][i]!=0:
        print(output[0][0][i])

[[203 650]
 [360 639]
 [364 690]
 [207 701]]
